# Learning Objectives
In this notebook, you will learn Spark Dataframe APIs.

# Question List

Solve the following questions using Spark Dataframe APIs

### Join

1. easy - https://pgexercises.com/questions/joins/simplejoin.html
2. easy - https://pgexercises.com/questions/joins/simplejoin2.html
3. easy - https://pgexercises.com/questions/joins/self2.html 
4. medium - https://pgexercises.com/questions/joins/threejoin.html (three join)
5. medium - https://pgexercises.com/questions/joins/sub.html (subquery and join)

### Aggregation

1. easy - https://pgexercises.com/questions/aggregates/count3.html Group by order by
2. easy - https://pgexercises.com/questions/aggregates/fachours.html group by order by
3. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth.html group by with condition 
4. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth2.html group by multi col
5. easy - https://pgexercises.com/questions/aggregates/members1.html count distinct
6. med - https://pgexercises.com/questions/aggregates/nbooking.html group by multiple cols, join

### String & Date

1. easy - https://pgexercises.com/questions/string/concat.html format string
2. easy - https://pgexercises.com/questions/string/case.html WHERE + string function
3. easy - https://pgexercises.com/questions/string/reg.html WHERE + string function
4. easy - https://pgexercises.com/questions/string/substr.html group by, substr
5. easy - https://pgexercises.com/questions/date/series.html generate ts
6. easy - https://pgexercises.com/questions/date/bookingspermonth.html extract month from ts

### Question

How can you produce a list of the start times for bookings by members named 'David Farrell'?

https://pgexercises.com/questions/joins/simplejoin.html

In [0]:
# Write you solution here
# hint: you might need to re-run `0 - ETL pgexercieses CSV files` notebook to init tables
from pyspark.sql.functions import col
df_bookings = spark.sql("select * from bookings")
df_members = spark.sql("select * from members")
df_member_bookings = df_bookings.join(df_members, df_bookings.memid == df_members.memid)
df_member_bookings.head()
display(df_member_bookings.select("firstname", "surname", "bookid", "starttime").where((col("surname") == "Farrell") & (col("firstname") == "David")))

firstname,surname,bookid,starttime
David,Farrell,3167,2012-09-18T09:00:00Z
David,Farrell,3172,2012-09-18T17:30:00Z
David,Farrell,3219,2012-09-18T13:30:00Z
David,Farrell,3229,2012-09-18T20:00:00Z
David,Farrell,3231,2012-09-19T09:30:00Z
David,Farrell,3233,2012-09-19T15:00:00Z
David,Farrell,3288,2012-09-19T12:00:00Z
David,Farrell,3335,2012-09-20T15:30:00Z
David,Farrell,3351,2012-09-20T11:30:00Z
David,Farrell,3356,2012-09-20T14:00:00Z


### Question
How can you produce a list of the start times for bookings for tennis courts, for the date '2012-09-21'? Return a list of start time and facility name pairings, ordered by the time.

[https://pgexercises.com/questions/joins/simplejoin2.html](https://pgexercises.com/questions/joins/simplejoin2.html)



In [0]:


df_bookings = spark.sql("select * from bookings b")
df_facilities = spark.sql("select * from facilities f")
df_facilities_bookings = df_bookings.join(df_facilities, df_bookings.facid == df_facilities.facid)
df_facilities_bookings.head()
display(df_facilities_bookings.select("name", "bookid", "starttime").filter(col("starttime").startswith("2012-09-21")).filter(col("name").startswith("Tennis")))

name,bookid,starttime
Tennis Court 1,3360,2012-09-21T08:00:00Z
Tennis Court 1,3361,2012-09-21T09:30:00Z
Tennis Court 1,3362,2012-09-21T12:00:00Z
Tennis Court 1,3363,2012-09-21T13:30:00Z
Tennis Court 1,3364,2012-09-21T15:30:00Z
Tennis Court 1,3365,2012-09-21T17:00:00Z
Tennis Court 2,3366,2012-09-21T08:00:00Z
Tennis Court 2,3367,2012-09-21T10:00:00Z
Tennis Court 2,3368,2012-09-21T11:30:00Z
Tennis Court 2,3369,2012-09-21T14:00:00Z


### Question
How can you output a list of all members, including the individual who recommended them (if any)? Ensure that results are ordered by (surname, firstname).

[https://pgexercises.com/questions/joins/self2.html](https://pgexercises.com/questions/joins/self2.html)



In [0]:
df_members1 = spark.sql("select * from members m1")
df_members2 = spark.sql("select * from members m2")
df_members_recommended = df_members1.join(df_members2, df_members1.recommendedby == df_members2.memid)
df_members_recommended.head()
display(df_members_recommended.select("m1.memid","m1.firstname", "m1.surname", col("m2.firstname").alias("recommended_firstname"), col("m2.surname").alias("recommended_surname")).orderBy([ "m1.surname", "m1.firstname"]))

memid,firstname,surname,recommended_firstname,recommended_surname
15,Florence,Bader,Ponder,Stibbons
12,Anne,Baker,Ponder,Stibbons
16,Timothy,Baker,Jemima,Farrell
8,Tim,Boothe,Tim,Rownam
5,Gerald,Butters,Darren,Smith
22,Joan,Coplin,Timothy,Baker
36,Erica,Crumpet,Tracy,Smith
7,Nancy,Dare,Janice,Joplette
20,Matthew,Genting,Gerald,Butters
35,John,Hunt,Millicent,Purview


### Question
How can you produce a list of all members who have used a tennis court? Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name followed by the facility name.

[https://pgexercises.com/questions/joins/threejoin.html](https://pgexercises.com/questions/joins/threejoin.html)



In [0]:
from pyspark.sql.functions import concat, lit

df_facilities = spark.sql("select * from facilities")
df_bookings = spark.sql("select * from bookings")
df_members = spark.sql("select * from members")

df_bookings_facilities = df_bookings.join(df_facilities, df_bookings.facid == df_facilities.facid)
df_bookings_facilities_members = df_bookings_facilities.join(df_members, df_bookings_facilities.memid == df_members.memid)
df_bookings_facilities_members = df_bookings_facilities_members.withColumn("member", concat(col("firstname"), lit(" ") ,col("surname")))
display(df_bookings_facilities_members.select("member", col("name").alias("facility")).dropDuplicates().filter(col("facility").startswith("Tennis")).orderBy(["member", "facility"]))

member,facility
Anne Baker,Tennis Court 1
Anne Baker,Tennis Court 2
Burton Tracy,Tennis Court 1
Burton Tracy,Tennis Court 2
Charles Owen,Tennis Court 1
Charles Owen,Tennis Court 2
Darren Smith,Tennis Court 2
David Farrell,Tennis Court 1
David Farrell,Tennis Court 2
David Jones,Tennis Court 1


### Question
How can you output a list of all members, including the individual who recommended them (if any), without using any joins? Ensure that there are no duplicates in the list, and that each firstname + surname pairing is formatted as a column and ordered.

[https://pgexercises.com/questions/joins/sub.html](https://pgexercises.com/questions/joins/sub.html)

PySpark can’t use that kind of subquery because it would need to run the inner query once for every row, and Spark is designed to process data all at once in parallel, not row-by-row. (`Unsupported subquery expression: Correlated scalar subqueries must be aggregated to return at most one row. SQLSTATE: 0A000`)

### Question
Produce a count of the number of recommendations each member has made. Order by member ID.

[https://pgexercises.com/questions/aggregates/count3.html](https://pgexercises.com/questions/aggregates/count3.html)



In [0]:
from pyspark.sql.functions import count
df_recommended_count = spark.sql("select * from members").groupBy("recommendedby").agg(count("recommendedby").alias("count")).select("recommendedby", "count").orderBy("recommendedby")
display(df_recommended_count)


recommendedby,count
null,0
1,5
2,3
3,1
4,2
5,1
6,1
9,2
11,1
13,2


### Question
Produce a list of the total number of slots booked per facility. For now, just produce an output table consisting of facility id and slots, sorted by facility id.

[https://pgexercises.com/questions/aggregates/fachours.html](https://pgexercises.com/questions/aggregates/fachours.html)



In [0]:
from pyspark.sql.functions import sum
df_fac_slots = spark.sql("select * from bookings").groupBy("facid").agg(sum("slots").alias("Total Slots")).select("facid", "Total Slots").orderBy("facid")
display(df_fac_slots)

facid,Total Slots
0,1320
1,1278
2,1209
3,830
4,1404
5,228
6,1104
7,908
8,911


### Question
Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots.

[https://pgexercises.com/questions/aggregates/fachoursbymonth.html](https://pgexercises.com/questions/aggregates/fachoursbymonth.html)



In [0]:
df_slot_month = spark.sql("select * from bookings").where(col("starttime").startswith("2012-09")).groupBy("facid").agg(sum("slots").alias("Total Slots")).select("facid",  "Total Slots").orderBy("Total Slots")
display(df_slot_month)

facid,Total Slots
5,122
3,422
7,426
8,471
6,540
2,570
1,588
0,591
4,648


### Question
Produce a list of the total number of slots booked per facility per month in the year of 2012. Produce an output table consisting of facility id and slots, sorted by the id and month.

[https://pgexercises.com/questions/aggregates/fachoursbymonth2.html](https://pgexercises.com/questions/aggregates/fachoursbymonth2.html)



In [0]:
from pyspark.sql.functions import substring
df_slot_by_month = spark.sql("select * from bookings").filter(col("starttime").startswith("2012")).withColumn("month", substring(col("starttime"), 6, 2)).groupBy("facid", "month").agg(sum("slots").alias("Total Slots")).select("facid", "month", "Total Slots").orderBy("facid", "month")
display(df_slot_by_month)

facid,month,Total Slots
0,07,270
0,08,459
0,09,591
1,07,207
1,08,483
1,09,588
2,07,180
2,08,459
2,09,570
3,07,104


### Question
Find the total number of members (including guests) who have made at least one booking.

[https://pgexercises.com/questions/aggregates/members1.html](https://pgexercises.com/questions/aggregates/members1.html)



In [0]:
df_bookings = spark.sql("select * from bookings b")
df_members = spark.sql("select * from members m")
df_member_bookings = df_bookings.join(df_members, df_bookings.memid == df_members.memid)
df_mem_bkngs = df_member_bookings.groupBy("m.memid").agg(count("m.memid").alias("Total Bookings")).select("m.memid", "Total Bookings").where(col("Total Bookings") > 0).agg(count("m.memid").alias("count"))
display(df_mem_bkngs)

count
30


### Question
Produce a list of each member name, id, and their first booking after September 1st 2012. Order by member ID.

[https://pgexercises.com/questions/aggregates/nbooking.html](https://pgexercises.com/questions/aggregates/nbooking.html)



In [0]:

from pyspark.sql.functions import min

df_bookings = spark.sql("select * from bookings b")
df_members = spark.sql("select * from members m")
df_member_bookings = df_bookings.join(df_members, "memid").filter(col("starttime") > "2012-09-01").groupBy("surname", "firstname", "memid").agg(min("starttime").alias("starttime")).orderBy("memid")
display(df_member_bookings)

surname,firstname,memid,starttime
GUEST,GUEST,0,2012-09-01T08:00:00Z
Smith,Darren,1,2012-09-01T09:00:00Z
Smith,Tracy,2,2012-09-01T11:30:00Z
Rownam,Tim,3,2012-09-01T16:00:00Z
Joplette,Janice,4,2012-09-01T15:00:00Z
Butters,Gerald,5,2012-09-02T12:30:00Z
Tracy,Burton,6,2012-09-01T15:00:00Z
Dare,Nancy,7,2012-09-01T12:30:00Z
Boothe,Tim,8,2012-09-01T08:30:00Z
Stibbons,Ponder,9,2012-09-01T11:00:00Z


### Question
Output the names of all members, formatted as 'Surname, Firstname'

[https://pgexercises.com/questions/string/concat.html](https://pgexercises.com/questions/string/concat.html)



In [0]:
df_members = spark.sql("select * from members")
display(df_members.withColumn("name", concat(col("firstname"), lit(", "), col("surname"))).select("name"))

name
"GUEST, GUEST"
"Darren, Smith"
"Tracy, Smith"
"Tim, Rownam"
"Janice, Joplette"
"Gerald, Butters"
"Burton, Tracy"
"Nancy, Dare"
"Tim, Boothe"
"Ponder, Stibbons"


### Question
Perform a case-insensitive search to find all facilities whose name begins with 'tennis'. Retrieve all columns.

[https://pgexercises.com/questions/string/case.html](https://pgexercises.com/questions/string/case.html)



In [0]:
from pyspark.sql.functions import upper
df_facilities = spark.sql("select * from facilities")
display(df_facilities.filter(upper(col("name")).like("TENNIS%")))

facid,name,membercost,guestcost,initialoutlay,monthlymaintenance
0,Tennis Court 1,5.0,25.0,10000,200
1,Tennis Court 2,5.0,25.0,8000,200


### Question
You've noticed that the club's member table has telephone numbers with very inconsistent formatting. You'd like to find all the telephone numbers that contain parentheses, returning the member ID and telephone number sorted by member ID.

[https://pgexercises.com/questions/string/reg.html](https://pgexercises.com/questions/string/reg.html)



In [0]:
from pyspark.sql.functions import rlike
df_members = spark.sql("select memid, telephone from members").filter((col("telephone").rlike( "\\(.*\\)")))
display(df_members)

memid,telephone
0,(000) 000-0000
3,(844) 693-0723
4,(833) 942-4710
5,(844) 078-4130
6,(822) 354-9973
7,(833) 776-4001
8,(811) 433-2547
9,(833) 160-3900
10,(855) 542-5251
11,(844) 536-8036


### Question
You'd like to produce a count of how many members you have whose surname starts with each letter of the alphabet. Sort by the letter, and don't worry about printing out a letter if the count is 0.

[https://pgexercises.com/questions/string/substr.html](https://pgexercises.com/questions/string/substr.html)



In [0]:
df_members = spark.sql("select * from members")
df_member_letters = df_members.withColumn("letter", substring(col("surname"), 1,1)).groupBy("letter").agg(count("letter").alias("count")).orderBy("letter")
display(df_member_letters)

letter,count
B,5
C,2
D,1
F,2
G,2
H,1
J,3
M,1
O,1
P,2


### Question
Produce a list of all the dates in October 2012. They can be output as a timestamp (with time set to midnight) or a date.

[https://pgexercises.com/questions/date/series.html](https://pgexercises.com/questions/date/series.html)



In [0]:
from pyspark.sql.functions import sequence, explode, to_date, lit

df = spark.sql("SELECT 1")
result = df.select(
    explode(
        sequence(
            to_date(lit("2012-10-01")),
            to_date(lit("2012-10-31"))
        )
    ).alias("ts")
)

display(result)

ts
2012-10-01
2012-10-02
2012-10-03
2012-10-04
2012-10-05
2012-10-06
2012-10-07
2012-10-08
2012-10-09
2012-10-10


### Question
Return a count of bookings for each month, sorted by month

[https://pgexercises.com/questions/date/bookingspermonth.html](https://pgexercises.com/questions/date/bookingspermonth.html)



In [0]:
from pyspark.sql.functions import date_trunc
df_bookings_month = spark.sql("select * from bookings").withColumn("month", date_trunc("month", col("starttime"))).groupBy("month").agg(count("month").alias("count")).orderBy("month")
display(df_bookings_month)


month,count
2012-07-01T00:00:00Z,658
2012-08-01T00:00:00Z,1472
2012-09-01T00:00:00Z,1913
2013-01-01T00:00:00Z,1
